# 05 · Appendix B.2 · Anticipation portfolio, 2000–2025

> **Appendix B — not part of the headline results.** The project's research question is the S&P 500 index effect, studied in `02_post_inclusion.ipynb`. This notebook belongs to the earlier anticipation (logit) work and is kept as an optional appendix. No core-sample announcement date is verified, so its labels are based on effective dates and cannot separate anticipation from announcement. Its Compustat inputs are the current, restated vintage rather than point-in-time data. Within this notebook, "headline sample" refers to the appendix's own main sample, not to the project's headline results.

This notebook extends the anticipation test from `04_anticipation_baseline.ipynb` in both directions without changing the specification. Model E is still the baseline plus the S&P earnings criterion, and the portfolio is still the equal-weighted top 20 held for one month.

**Rules fixed before seeing returns**

- **Walk-forward estimation.** For each calendar year from 2000, the logit is fitted on formation dates from January 1990 through September of the previous year. Every training label is therefore known before that year starts.
- **Membership after 2022.** It is reconstructed from the December 2022 members plus the recorded additions and removals. Those files come partly from Wikipedia tables and notebook 03's S&P notices, so 2023–2025 stays provisional.
- **Reporting.** The headline sample ends in December 2022, when verified membership ends. Prediction ends in September 2022, because later three-month labels depend on 2023 additions. All samples are reported whatever their result. 2023–2025 is a separate robustness check, and its membership data is provisional: the removals list ends in August 2025.

Returns are gross. As in notebook 04, a security that joins is held to the end of the month rather than sold at the effective-date close. The rule S&P's earnings criterion and the Compustat values use current definitions and data, which may differ from what was known at the time.

In [ ]:
from pathlib import Path

from IPython.utils.capture import capture_output
import numpy as np
import pandas as pd
import statsmodels.api as sm

save_prepared = True
top_n = 20
newey_west_lags = 12
bootstrap_draws = 2000
bootstrap_block_months = 12
bootstrap_seed = 20261106

inventory_path = Path("01_inventory.ipynb")
if not inventory_path.is_file():
    inventory_path = Path("notebooks") / "01_inventory.ipynb"
with capture_output(stdout=False, stderr=False):
    get_ipython().run_line_magic("run", f'"{inventory_path.resolve().as_posix()}"')

training_start = pd.Timestamp("1990-01-01")
first_test_year = 2000
last_formation = pd.Timestamp("2025-11-30")
data_end = monthly_panel["mthcaldt"].max()

## Rebuild membership for 2023–2025

Notebook 01's membership stops in December 2022. This cell starts from its December 2022 members and applies dated changes in order. A removal is matched by ticker within ±10 days and must refer to a current member. The removals file ends in August 2025, so securities removed after that date stay in the count. They are excluded from candidates, which matters little because removed securities are rarely among the largest non-members. A stock's status changes on its effective date. Results that use this reconstruction are labelled provisional and are not part of the headline sample.

In [ ]:
last_member_date = monthly_panel.loc[monthly_panel["mthcaldt"].le(history_end), "mthcaldt"].max()
current_members = set(monthly_panel.loc[
    monthly_panel["mthcaldt"].eq(last_member_date) & monthly_panel["in_sp500"], "permno"
])
starting_count = len(current_members)

additions_after = recent_additions.loc[
    recent_additions["permno"].notna() & recent_additions["effective_date"].gt(history_end), ["permno", "effective_date", "ticker"]
].astype({"permno": "int64"})
removal_candidates = ticker_candidates(recent_removals, tolerance_days=10).groupby("event_id")["permno"].agg(set)
change_list = pd.concat([
    additions_after.assign(kind="add", candidates=[{permno} for permno in additions_after["permno"]]),
    recent_removals.assign(kind="remove", candidates=recent_removals.index.to_series().map(removal_candidates))
    [["effective_date", "ticker", "kind", "candidates"]],
], ignore_index=True).sort_values(["effective_date", "kind"], ignore_index=True)

month_ends = sorted(monthly_panel.loc[monthly_panel["mthcaldt"].gt(last_member_date), "mthcaldt"].unique())
membership_rows, unresolved, next_change = [], [], 0
for month_end in month_ends:
    while next_change < len(change_list) and change_list.loc[next_change, "effective_date"] <= month_end:
        change = change_list.loc[next_change]
        if change["kind"] == "add":
            current_members.add(next(iter(change["candidates"])))
        else:
            matches = (change["candidates"] if isinstance(change["candidates"], set) else set()) & current_members
            if len(matches) == 1:
                current_members -= matches
            else:
                unresolved.append((change["effective_date"].date(), change["ticker"], len(matches)))
        next_change += 1
    membership_rows.append(pd.DataFrame({"permno": sorted(current_members), "mthcaldt": month_end}))
later_members = pd.concat(membership_rows, ignore_index=True).assign(member_after_2022=True)

monthly_panel = monthly_panel.merge(later_members, on=["permno", "mthcaldt"], how="left", validate="one_to_one")
monthly_panel["in_sp500_extended"] = monthly_panel["in_sp500"] | monthly_panel["member_after_2022"].eq(True)
member_counts = monthly_panel.loc[monthly_panel["mthcaldt"].ge(last_member_date)].groupby("mthcaldt")["in_sp500_extended"].sum()

print(f"December 2022 members: {starting_count}")
print(f"Changes applied: {next_change} of {len(change_list)}")
print(f"Removals that could not be resolved: {unresolved}")
display(member_counts.iloc[::6].rename("members with a CRSP row").to_frame())
print(f"Lowest and highest count: {member_counts.min()} and {member_counts.max()}")
if member_counts.min() < 490 or member_counts.max() > 520:
    raise ValueError("The reconstructed membership is outside a plausible range.")

## Rebuild the feature table through 2025

These are notebook 01's feature definitions, applied to the extended membership. The check compares every row through December 2022 with notebook 01's `features`, and stops the notebook if they differ. Labels get the same check.

In [ ]:
panel = monthly_panel
security = panel["permno"]
month_number = panel["mthcaldt"].dt.year * 12 + panel["mthcaldt"].dt.month
with np.errstate(divide="ignore"):
    log_return = np.log1p(panel["mthret"])


def months_since(lag):
    return month_number - month_number.groupby(security).shift(lag)


def trailing(values, window, statistic):
    rolled = getattr(values.groupby(security).rolling(window, min_periods=window), statistic)()
    return rolled.droplevel(0).where(months_since(window - 1).eq(window - 1))


price = panel["mthprc"].abs()
dollar_volume = panel["mthvol"] * price
raw = pd.DataFrame({
    "permno": security, "mthcaldt": panel["mthcaldt"],
    "log_mktcap": np.log(panel["mthcap"].where(panel["mthcap"].gt(0))),
    "dollar_volume_to_mktcap": dollar_volume / (panel["mthcap"] * 1000),
    "log_dollar_volume": np.log(dollar_volume.where(dollar_volume.gt(0))),
    "mom_12_1": np.expm1(trailing(log_return, 11, "sum").groupby(security).shift(1).where(months_since(1).eq(1))),
    "ret_1m": panel["mthret"], "ret_3m": np.expm1(trailing(log_return, 3, "sum")),
    "vol_12m": trailing(panel["mthret"], 12, "std"),
    "log_price": np.log(price.where(price.gt(0))),
})
extended = raw.loc[
    ~panel["in_sp500_extended"] & panel["eligible"] & panel["mthcaldt"].between(sample_start, data_end)
].reset_index(drop=True)
extended.insert(3, "mktcap_pct_rank", extended.groupby("mthcaldt")["log_mktcap"].rank(pct=True))
by_month = extended.groupby("mthcaldt")[winsorized_columns]
clipped = extended[feature_columns].copy()
clipped[winsorized_columns] = extended[winsorized_columns].clip(
    lower=by_month.transform(lambda column: column.quantile(0.01)),
    upper=by_month.transform(lambda column: column.quantile(0.99)),
)
clipped_by_month = clipped.groupby(extended["mthcaldt"])
extended[feature_columns] = (clipped - clipped_by_month.transform("mean")) / clipped_by_month.transform("std")
del raw, clipped

overlap = extended.loc[extended["mthcaldt"].le(history_end)].merge(
    features, on=["permno", "mthcaldt"], how="outer", suffixes=("", "_01"), indicator=True, validate="one_to_one"
)
if not overlap["_merge"].eq("both").all():
    raise ValueError("The rebuilt candidate rows differ from notebook 01.")
largest_gap = max(
    (overlap[column] - overlap[f"{column}_01"]).abs().max() for column in feature_columns
)
if largest_gap > 1e-9:
    raise ValueError(f"The rebuilt features differ from notebook 01 by {largest_gap}.")
print(f"Rows through 2022 matched with notebook 01: {len(overlap):,}; largest feature gap: {largest_gap:.1e}")

clean_after = events.loc[
    events["permno"].notna() & events["effective_date"].gt(history_end) & ~events["likely_spin_or_merger"],
    ["permno", "effective_date"],
].astype({"permno": "int64"})
all_targets = pd.concat([label_targets, clean_after], ignore_index=True)
if all_targets.duplicated().any():
    raise ValueError("An addition appears twice in the label targets.")
all_targets = all_targets.sort_values("effective_date", ignore_index=True)
extended = pd.merge_asof(
    extended.sort_values("mthcaldt", kind="stable"), all_targets,
    left_on="mthcaldt", right_on="effective_date", by="permno", direction="forward", allow_exact_matches=False,
).rename(columns={"effective_date": "next_effective", "mthcaldt": "formation_date"})
extended["lead_months"] = (
    (extended["next_effective"].dt.year - extended["formation_date"].dt.year) * 12
    + extended["next_effective"].dt.month - extended["formation_date"].dt.month
)
horizon_known = extended["formation_date"].dt.to_period("M") + 3 <= data_end.to_period("M")
extended["y3"] = extended["lead_months"].le(3).astype(float).where(horizon_known)

label_check = extended.merge(
    features.rename(columns={"mthcaldt": "formation_date", "y3": "y3_01"})[["permno", "formation_date", "y3_01"]],
    on=["permno", "formation_date"], validate="one_to_one",
).dropna(subset=["y3_01"])
if not label_check["y3"].eq(label_check["y3_01"].astype(float)).all():
    raise ValueError("The extended labels differ from notebook 01.")
print(f"Labels through 2022 matched with notebook 01: {len(label_check):,}")

## Candidate table

The same steps as notebook 04: remove securities whose firm already has another share class in the index, rank size after that, and add the earnings criterion with a four-month lag.

In [ ]:
firm_ids = pd.read_csv(
    paths["crsp_monthly"], usecols=["PERMNO", "PERMCO", "MthCalDt"], parse_dates=["MthCalDt"]
).rename(columns={"PERMNO": "permno", "PERMCO": "permco", "MthCalDt": "formation_date"}).drop_duplicates()
if firm_ids.duplicated(["permno", "formation_date"]).any():
    raise ValueError("A security has more than one PERMCO in a month.")
member_firms = monthly_panel.loc[monthly_panel["in_sp500_extended"], ["permno", "mthcaldt"]].rename(
    columns={"mthcaldt": "formation_date"}
).merge(firm_ids, on=["permno", "formation_date"], validate="one_to_one")[["permco", "formation_date"]].drop_duplicates().assign(firm_in_sp500=True)

table = extended.loc[extended["formation_date"].ge(training_start)].merge(
    firm_ids, on=["permno", "formation_date"], how="left", validate="one_to_one"
).merge(member_firms, on=["permco", "formation_date"], how="left", validate="many_to_one")
if table["permco"].isna().any():
    raise ValueError("A candidate row has no PERMCO.")
table = table.loc[~table["firm_in_sp500"].eq(True)].drop(columns="firm_in_sp500").reset_index(drop=True)
table["size_rank"] = table.groupby("formation_date")["mktcap_pct_rank"].rank(ascending=False, method="min")
table["log_size_rank"] = np.log(table["size_rank"])

quarterly = pd.read_csv(
    paths["comp_quarterly"], usecols=["gvkey", "datadate", "ibq"], parse_dates=["datadate"]
).dropna(subset=["ibq"]).drop_duplicates(["gvkey", "datadate", "ibq"])
quarterly = quarterly.loc[~quarterly.duplicated(["gvkey", "datadate"], keep=False)].sort_values(["gvkey", "datadate"], ignore_index=True)
quarterly["quarter_month"] = quarterly["datadate"].dt.year * 12 + quarterly["datadate"].dt.month
by_firm = quarterly.groupby("gvkey")
quarterly["earnings_4q"] = by_firm["ibq"].rolling(4, min_periods=4).sum().droplevel(0)
quarterly["consecutive_4q"] = quarterly["quarter_month"] - by_firm["quarter_month"].shift(3) == 9
quarterly["passes_rule"] = quarterly["ibq"].gt(0) & quarterly["earnings_4q"].gt(0)
availability_lag_months = 4
quarterly["known_month"] = quarterly["quarter_month"] + availability_lag_months

links = pd.read_csv(paths["ccm_link"], usecols=["gvkey", "LPERMNO", "LINKPRIM", "LINKDT", "LINKENDDT"])
links = links.loc[links["LINKPRIM"].isin(["P", "C"])].rename(columns={"LPERMNO": "permno"}).copy()
links["link_start"] = pd.to_datetime(links["LINKDT"])
links["link_end"] = pd.to_datetime(links["LINKENDDT"].replace("E", "2099-12-31"))
linked = table[["permno", "formation_date"]].merge(links[["permno", "gvkey", "link_start", "link_end"]], on="permno")
linked = linked.loc[linked["formation_date"].between(linked["link_start"], linked["link_end"]), ["permno", "formation_date", "gvkey"]]
linked = linked.loc[~linked.duplicated(["permno", "formation_date"], keep=False)]
linked["formation_month"] = linked["formation_date"].dt.year * 12 + linked["formation_date"].dt.month
matched = pd.merge_asof(
    linked.sort_values("formation_month", kind="stable"),
    quarterly.sort_values("known_month", kind="stable")[["gvkey", "known_month", "quarter_month", "consecutive_4q", "passes_rule"]],
    left_on="formation_month", right_on="known_month", by="gvkey", direction="backward",
)
if (matched["formation_month"] - matched["quarter_month"]).lt(availability_lag_months).any():
    raise ValueError("An earnings quarter was used before its assumed availability date.")
usable = matched["consecutive_4q"].eq(True) & (matched["formation_month"] - matched["quarter_month"]).le(12)
matched["earnings_status"] = np.where(usable, np.where(matched["passes_rule"].eq(True), "pass", "fail"), "unknown")
table = table.merge(
    matched[["permno", "formation_date", "earnings_status"]], on=["permno", "formation_date"], how="left", validate="one_to_one"
)
table["earnings_status"] = table["earnings_status"].fillna("unknown")
table["earnings_pass"] = table["earnings_status"].eq("pass").astype(int)
table["earnings_unknown"] = table["earnings_status"].eq("unknown").astype(int)

display(table.groupby(table["formation_date"].dt.year // 10 * 10).agg(
    rows=("permno", "size"), positive_rows=("y3", "sum"),
    earnings_unknown=("earnings_unknown", "mean"), earnings_pass=("earnings_pass", "mean"),
).round(3).rename_axis("decade"))

## Walk-forward estimation

Coefficients are refitted each January and applied to that calendar year's formation dates only. The table shows how the fit changes over time.

In [ ]:
liquidity_inputs = ["log_size_rank", "log_dollar_volume", "dollar_volume_to_mktcap"]
model_inputs = {
    "E: baseline + earnings": [*liquidity_inputs, "earnings_pass", "earnings_unknown"],
    "B: baseline": liquidity_inputs,
    "S: size rank": ["log_size_rank"],
}
for name in model_inputs:
    table[name] = np.nan
fit_paths = {}
for year in range(first_test_year, last_formation.year + 1):
    fit_end = pd.Timestamp(year - 1, 9, 30)
    apply_rows = table["formation_date"].dt.year.eq(year) & table["formation_date"].le(last_formation)
    for name, inputs in model_inputs.items():
        train = table.loc[table["formation_date"].le(fit_end) & table["y3"].notna()].dropna(subset=inputs)
        if train["formation_date"].max() > fit_end:
            raise ValueError("A training label is not known before the test year.")
        fit = sm.Logit(train["y3"], sm.add_constant(train[inputs])).fit(disp=False, maxiter=100)
        if not fit.mle_retvals["converged"]:
            raise RuntimeError(f"The logit for {name} in {year} did not converge.")
        scorable = apply_rows & table[inputs].notna().all(axis=1)
        table.loc[scorable, name] = fit.predict(sm.add_constant(table.loc[scorable, inputs], has_constant="add"))
        if name == "E: baseline + earnings":
            fit_paths[year] = {**fit.params.to_dict(), "training_rows": int(fit.nobs), "positive_rows": int(train["y3"].sum())}
display(pd.DataFrame(fit_paths).T.iloc[::5].round(3).rename_axis("model E, test year"))

evaluation = table.loc[table["formation_date"].dt.year.ge(first_test_year) & table["formation_date"].le(last_formation)].copy()
evaluation["largest_100"] = -evaluation["size_rank"]


def select_top(rows, score, n=top_n):
    return (
        rows.dropna(subset=[score])
        .sort_values(["formation_date", score, "permno"], ascending=[True, False, True], kind="stable")
        .groupby("formation_date").head(n)
    )


portfolio_lists = {name: select_top(evaluation, name) for name in model_inputs}
portfolio_lists["Largest 100 non-members"] = select_top(evaluation, "largest_100", n=100)
main_name = "E: baseline + earnings"
if portfolio_lists[main_name].groupby("formation_date").size().ne(top_n).any():
    raise ValueError("A monthly list does not contain 20 securities.")

## Prediction by period

Headline labels run through September 2022. Labels from October 2022 to September 2025 depend on provisional additions and are reported separately. The bootstrap is the same as in notebook 04.

In [ ]:
def block_bootstrap_ci(monthly_values):
    values = monthly_values.dropna().to_numpy()
    generator = np.random.default_rng(bootstrap_seed)
    blocks_needed = int(np.ceil(len(values) / bootstrap_block_months))
    starts = generator.integers(0, len(values) - bootstrap_block_months + 1, (bootstrap_draws, blocks_needed))
    offsets = starts[:, :, None] + np.arange(bootstrap_block_months)
    means = values[offsets].reshape(bootstrap_draws, -1)[:, :len(values)].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])


verified_label_end = pd.Timestamp("2022-09-30")  # Later labels depend on provisional 2023 additions.
prediction_periods = {
    "Headline 2000–09/2022": (2000, 2022), "2000–2009": (2000, 2009), "2010–2017": (2010, 2017),
    "2018–09/2022": (2018, 2022), "2013–09/2022": (2013, 2022),
    "Robustness 10/2022–09/2025 (provisional)": (2022, 2025),
}
prediction_table = {}
for period, (first, last) in prediction_periods.items():
    in_period = evaluation["formation_date"].dt.year.between(first, last)
    verified = evaluation["formation_date"].le(verified_label_end)
    rows = evaluation.loc[in_period & evaluation["y3"].notna() & (~verified if period.startswith("Robustness") else verified)]
    for name in model_inputs:
        selected = select_top(rows, name)
        monthly = selected.groupby("formation_date")["y3"].mean()
        low, high = block_bootstrap_ci(monthly)
        event_key = ["permno", "next_effective"]
        all_events = rows.loc[rows["y3"].eq(1), event_key].drop_duplicates()
        prediction_table[(period, name)] = {
            "months": len(monthly), "precision_at_20": monthly.mean(), "ci_low": low, "ci_high": high,
            "maximum_attainable": rows.groupby("formation_date")["y3"].sum().clip(upper=top_n).div(top_n).mean(),
            "event_recall": len(selected.loc[selected["y3"].eq(1), event_key].drop_duplicates()) / len(all_events),
        }
display(pd.DataFrame(prediction_table).T.rename_axis(["period", "model"]).round(4))

## Portfolio results

The headline sample runs from February 2000 to December 2022. Returns through December 2025 appear only in rows labelled as robustness checks. Newey–West standard errors use 12 lags on the calendar, so a missing month or an excluded year keeps its place in time, as in notebook 02. Model E is the main portfolio. *Post-publication* begins in 2021, after Bennett, Stulz, and Wang's 2020 paper, and the decay test estimates the change in alpha at that date.

In [ ]:
five_factors = read_monthly_factors(
    data_dir / "F-F_Research_Data_5_Factors_2x3_CSV.zip", ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]
)
momentum_factor = read_monthly_factors(data_dir / "F-F_Momentum_Factor_CSV.zip", ["Mom"])
monthly_factors = five_factors.join(momentum_factor, validate="one_to_one")
factor_columns = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]

market_returns = pd.read_csv(paths["index_monthly"], usecols=["mthcaldt", "vwretd"], parse_dates=["mthcaldt"])
market_returns = market_returns.assign(month=market_returns["mthcaldt"].dt.to_period("M")).set_index("month")["vwretd"]
stock_returns = monthly_panel[["permno", "mthcaldt", "mthret", "mthdelflg"]].copy()
stock_returns["return_month"] = stock_returns.pop("mthcaldt").dt.to_period("M")
stock_returns["stock_return"] = stock_returns.pop("mthret").mask(stock_returns.pop("mthdelflg").isin(["G", "M", "V"]))


def holding_returns(selected):
    holdings = selected[["permno", "formation_date", "y3", "lead_months"]].copy()
    holdings["return_month"] = holdings["formation_date"].dt.to_period("M") + 1
    return holdings.merge(stock_returns, on=["permno", "return_month"], how="left", validate="one_to_one")


def monthly_series(selected):
    holdings = holding_returns(selected)
    series = holdings.groupby("return_month").agg(
        positions=("permno", "size"), returns=("stock_return", "count"), portfolio=("stock_return", "mean")
    )
    series["portfolio"] = series["portfolio"].where(series["positions"].eq(series["returns"]))
    members = selected.groupby("formation_date")["permno"].agg(set)
    series["replaced_share"] = np.r_[np.nan, [len(now - before) / len(now) for before, now in zip(members.iloc[:-1], members.iloc[1:])]]
    series = series.join(market_returns.rename("market")).join(monthly_factors)
    series["excess"] = series["portfolio"] - series["RF"]
    series["market_adjusted"] = series["portfolio"] - series["market"]
    return series


def newey_west_ols(sample, outcome_column, regressors, lags=newey_west_lags):
    calendar = pd.period_range(sample.index.min(), sample.index.max(), freq="M")
    aligned = sample.reindex(calendar)
    observed = aligned[[outcome_column, *regressors]].notna().all(axis=1).to_numpy()
    outcome = aligned.loc[observed, outcome_column].to_numpy()
    design = np.column_stack([np.ones(observed.sum()), aligned.loc[observed, regressors].to_numpy()]) if regressors else np.ones((observed.sum(), 1))
    coefficients, _, rank, _ = np.linalg.lstsq(design, outcome, rcond=None)
    if rank < design.shape[1]:
        raise ValueError("The regression design is not full rank.")
    residuals = outcome - design @ coefficients
    scores = np.zeros((len(calendar), design.shape[1]))
    scores[observed] = design * residuals[:, None]
    middle = scores.T @ scores
    for lag in range(1, min(lags, len(calendar) - 1) + 1):
        cross = scores[lag:].T @ scores[:-lag]
        middle += (1 - lag / (lags + 1)) * (cross + cross.T)
    bread = np.linalg.inv(design.T @ design)
    covariance = bread @ middle @ bread * len(outcome) / (len(outcome) - design.shape[1])
    errors = np.sqrt(np.diag(covariance))
    names = ["const", *regressors]
    return pd.DataFrame({"coefficient": coefficients, "se": errors, "t": coefficients / errors}, index=names), residuals


portfolio_series = {name: monthly_series(selected) for name, selected in portfolio_lists.items()}
for name, series in portfolio_series.items():
    if series[factor_columns].isna().any().any():
        raise ValueError(f"Factors do not cover every return month for {name}.")
display(pd.DataFrame({
    name: {"return months": len(series), "blank months": int(series["portfolio"].isna().sum()),
           "first": str(series.index.min()), "last": str(series.index.max())}
    for name, series in portfolio_series.items()
}).T)

portfolio_samples = {
    "Headline 02/2000–2022": lambda i: i.year <= 2022, "02/2000–2009": lambda i: i.year <= 2009,
    "2010–2017": lambda i: (i.year >= 2010) & (i.year <= 2017), "2018–2022": lambda i: (i.year >= 2018) & (i.year <= 2022),
    "2013–2022": lambda i: (i.year >= 2013) & (i.year <= 2022), "Post-publication 2021–2022": lambda i: (i.year >= 2021) & (i.year <= 2022),
    "Without 2008": lambda i: (i.year <= 2022) & (i.year != 2008), "Without 2020": lambda i: (i.year <= 2022) & (i.year != 2020),
    "Robustness 2023–2025 (provisional)": lambda i: i.year >= 2023,
    "Robustness 02/2000–2025 (provisional after 2022)": lambda i: i.year >= 0,
    "Robustness post-publication 2021–2025 (provisional after 2022)": lambda i: i.year >= 2021,
}


def portfolio_summary(series):
    usable = series.dropna(subset=["portfolio"])
    alpha_table, residuals = newey_west_ols(series, "excess", factor_columns)
    adjusted_table, _ = newey_west_ols(series, "market_adjusted", [])
    annual_alpha = alpha_table.loc["const", "coefficient"] * 12
    annual_turnover = series["replaced_share"].mean() * 12
    cumulative = (1 + usable["portfolio"]).cumprod()
    return {
        "months": len(usable), "annual_return": cumulative.iloc[-1] ** (12 / len(usable)) - 1,
        "annual_market_return": (1 + usable["market"]).prod() ** (12 / len(usable)) - 1,
        "annual_volatility": usable["portfolio"].std() * np.sqrt(12),
        "sharpe": usable["excess"].mean() / usable["excess"].std() * np.sqrt(12),
        "market_adjusted_annual": adjusted_table.loc["const", "coefficient"] * 12,
        "market_adjusted_t": adjusted_table.loc["const", "t"],
        "alpha_annual": annual_alpha, "alpha_t": alpha_table.loc["const", "t"],
        "alpha_ci_low": annual_alpha - 1.96 * 12 * alpha_table.loc["const", "se"],
        "alpha_ci_high": annual_alpha + 1.96 * 12 * alpha_table.loc["const", "se"],
        "market_beta": alpha_table.loc["Mkt-RF", "coefficient"],
        "information_ratio": annual_alpha / (residuals.std() * np.sqrt(12)),
        "max_drawdown": (cumulative / cumulative.cummax() - 1).min(),
        "annual_one_way_turnover": annual_turnover,
        "break_even_bps": 1e4 * annual_alpha / (2 * annual_turnover),
    }


portfolio_results = pd.DataFrame({
    (sample, name): portfolio_summary(series.loc[keep(series.index)])
    for sample, keep in portfolio_samples.items() for name, series in portfolio_series.items()
}).T.rename_axis(["sample", "portfolio"])
display(portfolio_results.round(3))

main_series = portfolio_series[main_name]
main_headline = main_series.loc[main_series.index.year <= 2022]
full_table, _ = newey_west_ols(main_headline, "excess", factor_columns)
display(full_table.round(4).rename_axis("main portfolio, headline 02/2000–2022"))
for label, sample in {"headline through 2022": main_headline, "provisional through 2025": main_series}.items():
    decay_sample = sample.assign(post_2020=(sample.index.year >= 2021).astype(float))
    decay_table, _ = newey_west_ols(decay_sample, "excess", ["post_2020", *factor_columns])
    display(decay_table.loc[["const", "post_2020"]].round(4).rename_axis(f"alpha before 2021 and its change after, {label}"))

## Who earns the return?

As in notebook 04, this split uses outcomes known only afterward, so it describes the result and cannot be traded. Holdings without a complete three-month label are shown separately.

In [ ]:
main_holdings = holding_returns(portfolio_lists[main_name]).join(market_returns.rename("market"), on="return_month")
main_holdings["market_adjusted"] = main_holdings["stock_return"] - main_holdings["market"]
main_holdings["outcome"] = np.select(
    [main_holdings["y3"].isna(), main_holdings["y3"].eq(0)],
    ["label not complete", "does not join within 3 months"],
    default="joins in " + main_holdings["lead_months"].astype("Int64").astype(str) + " month(s)",
)
main_holdings["period"] = pd.cut(main_holdings["return_month"].dt.year, [1999, 2009, 2017, 2022, 2025], labels=["2000–2009", "2010–2017", "2018–2022", "2023–2025 (provisional)"])
display(main_holdings.groupby(["period", "outcome"], observed=True).agg(
    holdings=("permno", "size"), mean_market_adjusted=("market_adjusted", "mean"),
    median_market_adjusted=("market_adjusted", "median"),
).round(4))

list_rows = portfolio_lists[main_name].sort_values(
    ["formation_date", main_name, "permno"], ascending=[True, False, True]
)[["formation_date", "permno", "size_rank", "earnings_status", main_name]].rename(columns={main_name: "predicted_probability"})
if save_prepared:
    prepared_dir = data_dir / "prepared"
    prepared_dir.mkdir(parents=True, exist_ok=True)
    list_rows.to_csv(prepared_dir / "anticipation_extended_selections.csv", index=False)
print(f"Rows in the extended candidate list: {len(list_rows):,}")

## Is the provisional 2023–2025 alpha robust?

This diagnostic was added after seeing the 2023–2025 estimate, so it is a check on that number and not a separate test. It measures each security's contribution to the market-adjusted return, then re-estimates alpha without the largest contributors. Removing winners after the fact is deliberately harsh, and the result is not a tradable portfolio. It shows how many securities the number depends on. With 36 months and seven coefficients, Newey–West errors are also unreliable, so the second table varies the lag length.

In [ ]:
latest_tickers = names.sort_values("secinfostartdt").groupby("permno")["ticker"].last()
recent_holdings = main_holdings.loc[main_holdings["return_month"].dt.year.ge(2023)].copy()
recent_holdings["contribution"] = recent_holdings["market_adjusted"] / top_n
recent_months = recent_holdings["return_month"].nunique()
contributions = recent_holdings.groupby("permno").agg(
    months_held=("permno", "size"), annualized_contribution=("contribution", lambda values: values.sum() * 12 / recent_months),
).sort_values("annualized_contribution", ascending=False)
contributions.insert(0, "ticker", contributions.index.map(latest_tickers))
display(contributions.head(10).round(4))
print(f"Securities held in 2023–2025: {len(contributions)}")
print(f"Market-adjusted return per year, sum of contributions: {contributions['annualized_contribution'].sum():.4f}")

recent_factors = portfolio_series[main_name].loc[portfolio_series[main_name].index.year >= 2023]
excluded_results = {}
for removed in [0, 1, 3, 5, 10]:
    kept = recent_holdings.loc[~recent_holdings["permno"].isin(contributions.index[:removed])]
    returns = kept.groupby("return_month")["stock_return"].mean()
    sample = recent_factors.assign(excess=returns - recent_factors["RF"])
    alpha_table, _ = newey_west_ols(sample, "excess", factor_columns)
    excluded_results[removed] = {
        "alpha_annual": alpha_table.loc["const", "coefficient"] * 12, "alpha_t": alpha_table.loc["const", "t"],
    }
display(pd.DataFrame(excluded_results).T.round(3).rename_axis("largest contributors removed"))

lag_results = {}
for lags in [0, 3, 6, 12, 24]:
    alpha_recent, _ = newey_west_ols(recent_factors, "excess", factor_columns, lags=lags)
    alpha_full, _ = newey_west_ols(main_headline, "excess", factor_columns, lags=lags)
    lag_results[lags] = {"t, 2023–2025 (provisional)": alpha_recent.loc["const", "t"], "t, headline 02/2000–2022": alpha_full.loc["const", "t"]}
display(pd.DataFrame(lag_results).T.round(2).rename_axis("Newey–West lags"))

## Audit fix 2 · Headline sample ends in December 2022

**What changed.** Headline portfolio results now end in December 2022 and headline prediction ends in September 2022. Rows that use 2023–2025 data are labelled as robustness checks. The removals list ends in August 2025, so that membership is provisional.

**Why.** Membership is verified only through December 2022. Prediction stops three months earlier because later labels depend on 2023 additions.

| Portfolio sample | Model E: annual alpha | t | Model B: annual alpha | t |
| --- | --- | --- | --- | --- |
| **Headline 02/2000–2022** | 2.8% | 1.16 | 3.2% | 1.35 |
| 02/2000–2009 | 8.2% | 2.43 | 7.1% | 2.30 |
| 2010–2017 | 1.7% | 0.65 | 1.0% | 0.34 |
| **2018–2022** | 0.0% | −0.01 | 5.1% | 0.95 |
| Robustness 2023–2025 (provisional) | 18.2% | 3.10 | 7.2% | 0.65 |
| Robustness 02/2000–2025 (provisional after 2022) | 4.0% | 1.75 | 3.7% | 1.60 |

Headline precision at 20 for 2000–09/2022 is 9.1% for model E (95% interval 7.6–10.3) and 7.0% for model B (5.6–8.2). The maximum attainable is 24.3%.

**Effect.** Without the provisional months, model E's alpha falls from 4.0% (t = 1.75) to 2.8% (t = 1.16). Model B is higher than model E in the headline sample, but this difference is not tested. No headline estimate reaches t > 3.

**Decay test.** In the headline sample, model E's monthly alpha changes by −1.30 points after 2020 (t = −2.48), but that period has only 24 months. Through 2025 the change is +0.06 points (t = 0.09). The two results conflict, so neither establishes post-publication decay.